In [3]:
# Read parameters passed from Activator
# Toggle this cell as: "Parameter cell"

deviceId = ""
ts = ""
tempC = 0.0
humidity = 0.0
thresholdTemp = 30.0
thresholdHum = 75.0

StatementMeta(, 7fa562cd-3546-4eda-b8de-7bcd5e8e3df1, 5, Finished, Available, Finished)

In [4]:
import os, json, requests
from datetime import datetime, timezone, timedelta

# Format timestamp to UTC + 7

def to_utc7(ts_str: str):
    try:
        if ts_str and ts_str.strip():
            dt = datetime.fromisoformat(ts_str)
        else:
            # fallback: current UTC time
            dt = datetime.now(timezone.utc)
    except Exception:
        dt = datetime.now(timezone.utc)

    return dt.astimezone(
        timezone(timedelta(hours=7))
    ).strftime("%Y-%m-%d %H:%M:%S")

ts_local = to_utc7(ts)

# Secrets (recommended via notebookutils / secret store)
# Pseudocode – wire to your secret storage approach in Fabric:
# tg_token = notebookutils.credentials.getSecret("kv_scope","TG_BOT_TOKEN")
# tg_chat_id = notebookutils.credentials.getSecret("kv_scope","TG_CHAT_ID")
# foundry_key = notebookutils.credentials.getSecret("kv_scope","FOUNDRY_API_KEY")
# foundry_endpoint = notebookutils.credentials.getSecret("kv_scope","FOUNDRY_CHAT_ENDPOINT")

tg_token = "Your_Telegram_Bot_Token"
tg_chat_id = "Your_Telegram_Chat_ID"
foundry_endpoint = "Your_Microsoft_Foundry_LLM_Endpoint"
foundry_key = "Your_Microsoft_Foundry_LLM_API_Key"

# Call Azure AI Foundry

system = (
  "You are a concise operations assistant. "
  "Given a temperature/humidity alert, produce: "
  "Risk (1 line), Likely causes (1-2 bullets), Actions (max 3 bullets), Escalation (1 line). "
  "Do not invent sensor values."
)

user = {
  "deviceId": deviceId,
  "ts": ts,
  "tempC": tempC,
  "humidity": humidity,
  "thresholdTemp": thresholdTemp,
  "thresholdHum": thresholdHum
}

payload = {
  "messages": [
    {"role": "system", "content": system},
    {"role": "user", "content": f"Alert telemetry JSON: {json.dumps(user)}"}
  ],
  "temperature": 0.2,
  "max_tokens": 220
}

r = requests.post(
  foundry_endpoint,
  headers={"Content-Type":"application/json", "api-key": foundry_key},
  data=json.dumps(payload),
  timeout=60
)
r.raise_for_status()
llm_text = r.json()["choices"][0]["message"]["content"]


# Prepare Telegram message

msg = (
  f"🚨 ENV ALERT\n"
  f"Device: {deviceId}\n"
  f"Temp: {tempC:.1f}°C\n"
  f"Hum : {humidity:.1f}%\n"
  f"Time: {ts_local}\n\n"
  f"✅ Suggested actions:\n{llm_text}"
)


StatementMeta(, 7fa562cd-3546-4eda-b8de-7bcd5e8e3df1, 6, Finished, Available, Finished)

In [5]:
# Send Telegram message

tg_url = f"https://api.telegram.org/bot{tg_token}/sendMessage"
requests.post(tg_url, data={"chat_id": tg_chat_id, "text": msg}, timeout=30).raise_for_status()

print("Sent Telegram alert.")


StatementMeta(, 7fa562cd-3546-4eda-b8de-7bcd5e8e3df1, 7, Finished, Available, Finished)

Sent Telegram alert.
